In [25]:
import torch
import sys
sys.path.append("src")
from game.utils import Constants, Translator
from game.state import State
from game.game import Game


In [26]:
torch.save(Constants.initial_board(), "tests/fixtures/boards/init_board.pt")

In [27]:
state = State.initial_state()
torch.save([state.get_player_mask(False), state.get_player_mask(True)], "tests/fixtures/game_fixtures/player_masks.pt")
state.save("tests/fixtures/boards/init_board.json")

In [28]:
state.save("temp/test_state_saving.json")

In [29]:
State.from_file("temp/test_state_saving.json").state.shape

torch.Size([6, 4, 8])

In [30]:
from agents.mcts import Tree, Node

In [31]:
game = Game.new_game()
tree = Tree(game, Constants.BATCH_SIZE, 2)
policy_size = game.full_move_tensor(0).shape

tree.root.set(torch.rand(policy_size), 1)

In [32]:
torch.rand(policy_size).shape, game.full_move_tensor(0).shape

(torch.Size([4, 8, 33]), torch.Size([4, 8, 33]))

In [33]:
for _ in range(4096):
    end_node = tree.root.rollout()
    end_node.set(torch.rand(policy_size), 1)
    # print([(i, node) for i, node in enumerate(tree.root.children) if node is not None])
    root = end_node.backpropagate(end_node._parent_child_index)
    # print(root.visit_count)


KeyboardInterrupt: 

In [ ]:
tree.full_dot("temp/tree.dot")
tree.optimal_path_dot("temp/path.dot")

In [ ]:
x1, y1, index = game.all_moves()
x2, y2, flag = game.move_from_index(index)
moves = list(zip(x1, y1, x2, y2, flag))
tree.make_move(*moves[0])
tree.optimal_path_dot("temp/path_reduced.dot")

In [ ]:
import timeit
from typing import cast

stmt = "a = root.mean_value if isinstance(root, Node) else 0"  # or use "type(root) is Node"
# stmt = "a = cast(Node, root).mean_value"  # or use "type(root) is Node"
reps = 1_000_000

duration = timeit.timeit(stmt, globals=globals(), number=reps)
print(f"{stmt}: {duration:.4f} sec over {reps} runs, avg {duration/reps*1e6:.2f} μs/run")

a = root.mean_value if isinstance(root, Node) else 0: 0.0679 sec over 1000000 runs, avg 0.07 μs/run
